In [1]:
import pandas as pd

# Check Lahore pollution data first
lahore_poll = pd.read_csv(
    r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\Pollution_Data\historical_air_pollution_all_lahore.csv"
)

print(f"Shape: {lahore_poll.shape}")
print(f"\nColumns: {lahore_poll.columns.tolist()}")
print(f"\nFirst 3 rows:")
print(lahore_poll.head(3))
print(f"\nDate range:")
print(f"From: {lahore_poll.iloc[0, 0]}")
print(f"To:   {lahore_poll.iloc[-1, 0]}")

Shape: (34433, 10)

Columns: ['Timestamp', 'AQI', 'CO', 'NO', 'NO2', 'O3', 'SO2', 'PM2.5', 'PM10', 'NH3']

First 3 rows:
             Timestamp  AQI        CO      NO     NO2   O3    SO2   PM2.5  \
0  2022-01-01 00:00:00  5.0  11962.89  219.94  124.75  0.0  15.97  843.89   
1  2022-01-01 01:00:00  5.0  10040.28  173.45  106.93  0.0  10.25  752.14   
2  2022-01-01 02:00:00  5.0   7904.05  118.02   89.11  0.0   8.23  649.93   

      PM10    NH3  
0  1003.04  27.61  
1   889.02  16.21  
2   773.83  14.69  

Date range:
From: 2022-01-01 00:00:00
To:   2026-01-01 01:00:00


In [3]:
print("PM2.5 Statistics:")
print(lahore_poll['PM2.5'].describe())

print("\nPM10 Statistics:")
print(lahore_poll['PM10'].describe())

print("\nTimestamp range check:")
lahore_poll['Timestamp'] = pd.to_datetime(lahore_poll['Timestamp'])
print(f"Start: {lahore_poll['Timestamp'].min()}")
print(f"End:   {lahore_poll['Timestamp'].max()}")

# Check smog season coverage
smog = lahore_poll[lahore_poll['Timestamp'].dt.month.isin([11, 12, 1, 2])]
print(f"\nSmog season rows: {len(smog)}")

PM2.5 Statistics:
count    34433.000000
mean       174.210807
std        181.022199
min          0.000000
25%         53.570000
50%        102.890000
75%        231.260000
max       1591.190000
Name: PM2.5, dtype: float64

PM10 Statistics:
count    34433.000000
mean       217.220102
std        213.617157
min      -9999.000000
25%         76.920000
50%        138.560000
75%        291.580000
max       1724.080000
Name: PM10, dtype: float64

Timestamp range check:
Start: 2022-01-01 00:00:00
End:   2026-01-01 01:00:00

Smog season rows: 11356


In [5]:
# Check -9999 values
print(f"PM10 -9999 count: {(lahore_poll['PM10'] == -9999).sum()}")
print(f"PM2.5 zero count: {(lahore_poll['PM2.5'] == 0).sum()}")

# Filter only smog season: Nov, Dec, Jan, Feb
smog_poll = lahore_poll[
    lahore_poll['Timestamp'].dt.month.isin([11, 12, 1, 2])
].copy()

# Replace -9999 with NaN
smog_poll['PM10'] = smog_poll['PM10'].replace(-9999, pd.NA)

print(f"\nSmog season rows: {len(smog_poll)}")
print(f"PM2.5 missing: {smog_poll['PM2.5'].isna().sum()}")
print(f"PM10 missing after fix: {smog_poll['PM10'].isna().sum()}")
print(f"\nDate range in smog data:")
print(f"From: {smog_poll['Timestamp'].min()}")
print(f"To:   {smog_poll['Timestamp'].max()}")

PM10 -9999 count: 1
PM2.5 zero count: 3

Smog season rows: 11356
PM2.5 missing: 0
PM10 missing after fix: 1

Date range in smog data:
From: 2022-01-01 00:00:00
To:   2026-01-01 01:00:00


In [7]:
import os

cities = ['lahore', 'islamabad', 'faisalabad', 'multan']
poll_path = r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\Pollution_Data"

all_poll = []

for city in cities:
    filepath = os.path.join(poll_path, 
        f"historical_air_pollution_all_{city}.csv")
    df_city = pd.read_csv(filepath)
    df_city['city'] = city.capitalize()
    df_city['Timestamp'] = pd.to_datetime(df_city['Timestamp'])
    
    # Filter smog season only
    df_smog = df_city[
        df_city['Timestamp'].dt.month.isin([11, 12, 1, 2])
    ].copy()
    
    # Clean -9999
    df_smog['PM10'] = df_smog['PM10'].replace(-9999, pd.NA)
    df_smog['PM2.5'] = df_smog['PM2.5'].replace(0, pd.NA)
    
    all_poll.append(df_smog)
    print(f"{city.capitalize()}: {len(df_smog)} smog season rows")

pollution_df = pd.concat(all_poll, ignore_index=True)
print(f"\nTotal pollution rows: {len(pollution_df)}")
print(f"Columns: {pollution_df.columns.tolist()}")

Lahore: 11356 smog season rows
Islamabad: 11356 smog season rows
Faisalabad: 11357 smog season rows
Multan: 11356 smog season rows

Total pollution rows: 45425
Columns: ['Timestamp', 'AQI', 'CO', 'NO', 'NO2', 'O3', 'SO2', 'PM2.5', 'PM10', 'NH3', 'city']


In [9]:
# Load weather master dataset
weather_df = pd.read_csv(
    r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\master_dataset.csv"
)
weather_df['datetime'] = pd.to_datetime(weather_df['datetime'])

print(f"Weather data shape: {weather_df.shape}")
print(f"Weather date range: {weather_df['datetime'].min()} to {weather_df['datetime'].max()}")
print(f"\nWeather cities: {weather_df['city'].unique()}")

# Keep only PM2.5 and PM10 from pollution
pollution_clean = pollution_df[['Timestamp', 'city', 'PM2.5', 'PM10']].copy()
pollution_clean.columns = ['datetime', 'city', 'pm25', 'pm10']

print(f"\nPollution data shape: {pollution_clean.shape}")
print(f"Pollution date range: {pollution_clean['datetime'].min()} to {pollution_clean['datetime'].max()}")
print(f"\nPollution cities: {pollution_clean['city'].unique()}")

Weather data shape: (40976, 15)
Weather date range: 2021-11-01 00:00:00 to 2025-02-28 23:00:00

Weather cities: ['Lahore' 'Islamabad' 'Faisalabad' 'Multan']

Pollution data shape: (45425, 4)
Pollution date range: 2022-01-01 00:00:00 to 2026-01-01 02:00:00

Pollution cities: ['Lahore' 'Islamabad' 'Faisalabad' 'Multan']


In [11]:
# Merge weather + pollution on datetime + city
merged_df = pd.merge(
    weather_df,
    pollution_clean,
    left_on=['datetime', 'city'],
    right_on=['datetime', 'city'],
    how='inner'  # Only keep rows where both exist
)

print(f"After merge shape: {merged_df.shape}")
print(f"Columns: {merged_df.columns.tolist()}")
print(f"\nDate range: {merged_df['datetime'].min()} to {merged_df['datetime'].max()}")
print(f"\nCity counts:")
print(merged_df['city'].value_counts())
print(f"\nMissing values:")
print(merged_df[['pm25', 'pm10']].isnull().sum())

After merge shape: (35542, 17)
Columns: ['datetime', 'city', 'temp', 'humidity', 'dew', 'windspeed', 'windgust', 'winddir', 'sealevelpressure', 'cloudcover', 'visibility', 'precip', 'snow', 'snowdepth', 'month', 'pm25', 'pm10']

Date range: 2022-01-01 00:00:00 to 2025-02-28 23:00:00

City counts:
city
Lahore        9885
Multan        9869
Islamabad     8717
Faisalabad    7071
Name: count, dtype: int64

Missing values:
pm25    0
pm10    4
dtype: int64


In [13]:
# Fix PM10 missing values with median per city
merged_df['pm10'] = merged_df.groupby('city')['pm10'].transform(
    lambda x: x.fillna(x.median())
)

# Verify
print(f"Missing after fix: {merged_df[['pm25', 'pm10']].isnull().sum().sum()}")

# Save merged dataset
output_path = r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\master_dataset_with_pollution.csv"
merged_df.to_csv(output_path, index=False)

print(f"\nDataset saved!")
print(f"Final shape: {merged_df.shape}")
print(f"\nSample data:")
print(merged_df[['datetime', 'city', 'visibility', 'pm25', 'pm10']].head(5))

C:\Users\Admin\AppData\Local\Temp\ipykernel_17896\516501103.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.fillna(x.median())
C:\Users\Admin\AppData\Local\Temp\ipykernel_17896\516501103.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.fillna(x.median())
C:\Users\Admin\AppData\Local\Temp\ipykernel_17896\516501103.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, se

Missing after fix: 0

Dataset saved!
Final shape: (35542, 17)

Sample data:
             datetime    city  visibility    pm25     pm10
0 2022-01-01 00:00:00  Lahore         2.9  843.89  1003.04
1 2022-01-01 01:00:00  Lahore         2.9  752.14   889.02
2 2022-01-01 02:00:00  Lahore         1.9  649.93   773.83
3 2022-01-01 03:00:00  Lahore         1.0  549.10   658.18
4 2022-01-01 04:00:00  Lahore         1.0  481.60   571.24


In [15]:
print("PM2.5 Statistics by City:")
print(merged_df.groupby('city')['pm25'].describe().round(2))

print("\nPM10 Statistics by City:")
print(merged_df.groupby('city')['pm10'].describe().round(2))

# Check correlation with visibility
print("\nCorrelation with visibility:")
print(merged_df[['visibility', 'pm25', 'pm10']].corr().round(3))

PM2.5 Statistics by City:
             count    mean     std    min     25%     50%     75%      max
city                                                                      
Faisalabad  7071.0  223.19  150.88  11.83  106.24  188.48  300.76   892.35
Islamabad   8717.0  204.50  136.61   4.80  100.06  179.82  284.56   803.28
Lahore      9885.0  338.13  225.90  15.56  166.64  291.31  463.52  1591.19
Multan      9869.0  201.66  120.55   6.40  109.38  181.67  271.95   823.40

PM10 Statistics by City:
             count    mean     std    min     25%     50%     75%      max
city                                                                      
Faisalabad  7071.0  268.45  168.97  15.73  136.82  233.84  363.35   980.61
Islamabad   8717.0  240.39  155.72   5.26  120.96  214.91  333.03   915.24
Lahore      9885.0  399.17  254.70  18.59  205.84  350.64  537.52  1724.08
Multan      9869.0  252.55  140.93  12.69  143.87  231.84  338.02   963.10

Correlation with visibility:
            visibi

In [17]:
# Drop missing visibility rows
before = len(merged_df)
merged_df = merged_df.dropna(subset=['visibility'])
print(f"After dropping missing visibility: {len(merged_df)} rows")

# Remove visibility outliers
merged_df = merged_df[merged_df['visibility'] <= 48]
print(f"After outlier removal: {len(merged_df)} rows")

# Feature 1: Hour + smog prone flag
merged_df['hour'] = merged_df['datetime'].dt.hour
merged_df['is_smog_prone_hour'] = merged_df['hour'].apply(
    lambda h: 1 if 4 <= h <= 9 else 0)

# Feature 2: Dew point depression
merged_df['dew_point_depression'] = merged_df['temp'] - merged_df['dew']

# Feature 3: One-hot encoding
merged_df = pd.get_dummies(merged_df, columns=['city'], prefix='city')

print(f"\nShape after features: {merged_df.shape}")
print(f"Columns: {merged_df.columns.tolist()}")

After dropping missing visibility: 35542 rows
After outlier removal: 35542 rows

Shape after features: (35542, 23)
Columns: ['datetime', 'temp', 'humidity', 'dew', 'windspeed', 'windgust', 'winddir', 'sealevelpressure', 'cloudcover', 'visibility', 'precip', 'snow', 'snowdepth', 'month', 'pm25', 'pm10', 'hour', 'is_smog_prone_hour', 'dew_point_depression', 'city_Faisalabad', 'city_Islamabad', 'city_Lahore', 'city_Multan']


In [19]:
import numpy as np

# Reconstruct city_temp for groupby
city_cols = ['city_Faisalabad', 'city_Islamabad', 'city_Lahore', 'city_Multan']
merged_df['city_temp'] = merged_df[city_cols].idxmax(axis=1).str.replace('city_', '', regex=False)

# Sort by city and datetime
merged_df = merged_df.sort_values(['city_temp', 'datetime']).reset_index(drop=True)

# Lag feature — previous visibility reading
merged_df['visibility_prev_reading'] = merged_df.groupby('city_temp')['visibility'].shift(1)

# Time gap
merged_df['prev_datetime_temp'] = merged_df.groupby('city_temp')['datetime'].shift(1)
merged_df['time_gap_hours'] = (merged_df['datetime'] - merged_df['prev_datetime_temp']).dt.total_seconds() / 3600

# Drop helper columns
merged_df = merged_df.drop(columns=['city_temp', 'prev_datetime_temp'])

# Wind direction encoding
merged_df['winddir_sin'] = np.sin(np.radians(merged_df['winddir']))
merged_df['winddir_cos'] = np.cos(np.radians(merged_df['winddir']))

# Fix winddir outliers
merged_df = merged_df[merged_df['winddir'] <= 360]

# Filter unreliable lag values
merged_df = merged_df[merged_df['time_gap_hours'].isna() | (merged_df['time_gap_hours'] <= 2)]

print(f"Final shape: {merged_df.shape}")
print(f"Missing values:\n{merged_df.isnull().sum()[merged_df.isnull().sum() > 0]}")

Final shape: (32351, 27)
Missing values:
visibility_prev_reading    4
time_gap_hours             4
dtype: int64


In [21]:
# Drop 4 missing visibility_prev_reading rows
merged_df = merged_df.dropna(subset=['visibility_prev_reading'])

# Drop time_gap_hours (quality column — not needed for training)
merged_df = merged_df.drop(columns=['time_gap_hours'])

# Final verification
print(f"Final shape: {merged_df.shape}")
print(f"Missing values: {merged_df.isnull().sum().sum()}")
print(f"\nFinal columns ({len(merged_df.columns)}):")
print(merged_df.columns.tolist())

# Save
output_path = r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\master_dataset_final.csv"
merged_df.to_csv(output_path, index=False)
print(f"\nFinal dataset saved!")

Final shape: (32347, 26)
Missing values: 0

Final columns (26):
['datetime', 'temp', 'humidity', 'dew', 'windspeed', 'windgust', 'winddir', 'sealevelpressure', 'cloudcover', 'visibility', 'precip', 'snow', 'snowdepth', 'month', 'pm25', 'pm10', 'hour', 'is_smog_prone_hour', 'dew_point_depression', 'city_Faisalabad', 'city_Islamabad', 'city_Lahore', 'city_Multan', 'visibility_prev_reading', 'winddir_sin', 'winddir_cos']

Final dataset saved!


In [25]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
import numpy as np

# X and y
y = merged_df['visibility']
X = merged_df.drop(columns=['visibility', 'datetime'])

print(f"X shape: {X.shape}")
print(f"Features: {X.columns.tolist()}")

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"\nTraining: {X_train.shape}")
print(f"Testing:  {X_test.shape}")

# Train Random Forest
print("\nTraining Random Forest...")
rf_model = RandomForestRegressor(
    n_estimators=100, max_depth=20,
    min_samples_split=5, min_samples_leaf=2,
    random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

# Train XGBoost
print("Training XGBoost...")
xgb_model = xgb.XGBRegressor(
    n_estimators=100, max_depth=6,
    learning_rate=0.1, subsample=0.8,
    colsample_bytree=0.8, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

# Results
print("\n" + "="*40)
print("RESULTS COMPARISON")
print("="*40)
print(f"{'Metric':<10} {'Random Forest':>15} {'XGBoost':>15}")
print("-"*40)
print(f"{'R²':<10} {r2_score(y_test, rf_pred):>15.4f} {r2_score(y_test, xgb_pred):>15.4f}")
print(f"{'MAE':<10} {mean_absolute_error(y_test, rf_pred):>15.4f} {mean_absolute_error(y_test, xgb_pred):>15.4f}")
print(f"{'RMSE':<10} {np.sqrt(mean_squared_error(y_test, rf_pred)):>15.4f} {np.sqrt(mean_squared_error(y_test, xgb_pred)):>15.4f}")


X shape: (32347, 24)
Features: ['temp', 'humidity', 'dew', 'windspeed', 'windgust', 'winddir', 'sealevelpressure', 'cloudcover', 'precip', 'snow', 'snowdepth', 'month', 'pm25', 'pm10', 'hour', 'is_smog_prone_hour', 'dew_point_depression', 'city_Faisalabad', 'city_Islamabad', 'city_Lahore', 'city_Multan', 'visibility_prev_reading', 'winddir_sin', 'winddir_cos']

Training: (25877, 24)
Testing:  (6470, 24)

Training Random Forest...
Training XGBoost...

RESULTS COMPARISON
Metric       Random Forest         XGBoost
----------------------------------------
R²                  0.9382          0.9473
MAE                 0.8723          1.0700
RMSE                2.1765          2.0087


In [27]:
import joblib

joblib.dump(rf_model, 
    r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\rf_model_v2.pkl")
joblib.dump(xgb_model, 
    r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\xgb_model_v2.pkl")

print("Models saved!")
print("rf_model_v2.pkl ✅")
print("xgb_model_v2.pkl ✅")

Models saved!
rf_model_v2.pkl ✅
xgb_model_v2.pkl ✅
